# Chapter 17. Working with Instruction-Tuned LLMs

## 17.2 Learning Objectives

By the end of this chapter, you should be able to:

- distinguish a base language model from an instruction-tuned model;
- represent a conversation using `system`, `user`, and `assistant` messages;
- use system messages to influence model behavior;
- configure generation using parameters such as `max_new_tokens`, `temperature`, `top_p`, and repetition penalties;
- estimate the memory required for model parameters at different numerical precisions.

## 17.3 Base Models vs Instruction-Tuned Models

In [4]:
base_checkpoint = "HuggingFaceTB/SmolLM2-135M"
instruct_checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
# Base
base_tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
base_model = AutoModelForCausalLM.from_pretrained(base_checkpoint).to(device)
base_model.eval()
# Instruct
tokenizer = AutoTokenizer.from_pretrained(instruct_checkpoint)
model = AutoModelForCausalLM.from_pretrained(instruct_checkpoint).to(device)
model.eval()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (r

In [7]:
prompt = "Explain gradient descent in one short paragraph."

In [8]:
base_inputs = base_tokenizer(prompt, return_tensors="pt").to(device)

with torch.inference_mode():
    base_output = base_model.generate(
        **base_inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=base_tokenizer.eos_token_id,
    )

print(base_tokenizer.decode(base_output[0], skip_special_tokens=True))

Explain gradient descent in one short paragraph.

The gradient descent algorithm is a method of minimizing a function by iteratively adjusting the parameters of the function to minimize the cost function. The gradient descent algorithm is a simple algorithm that is used to minimize a function. The gradient descent algorithm is a simple algorithm that is used to minimize a function. The gradient descent algorithm is a simple algorithm that is used to minimize a function. The gradient descent algorithm


In [10]:
messages = [{"role": "user", "content": prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
Explain gradient descent in one short paragraph.
assistant
Gradient descent is a fundamental optimization technique used in machine learning to find the minimum of a function. It starts with a starting point, typically a random initial guess, and iteratively updates the solution to the problem by minimizing the difference between the current solution and the target function. This process is repeated until the gradient of the function is zero, indicating that the solution is optimal.


In [11]:
prompt_length = inputs["input_ids"].shape[-1]
response_ids = output[0, prompt_length:]
response = tokenizer.decode(response_ids, skip_special_tokens=True)
print(response)

Gradient descent is a fundamental optimization technique used in machine learning to find the minimum of a function. It starts with a starting point, typically a random initial guess, and iteratively updates the solution to the problem by minimizing the difference between the current solution and the target function. This process is repeated until the gradient of the function is zero, indicating that the solution is optimal.


## 17.4 Chat Templates

Modern chat models commonly assume a conversation is stored as a list of dictionaries, where every message contains two basic pieces of information: role and content. The content is the actual text of the message. The role identifies who's the author of the content. Hugging Face chat models commonly use the roles `system`, `user`, and `assistant`.

A user message represents input from the person interacting with the model, and an assistant message represents a response generated—or expected to be generated—by the model. A system message provides higher-level instructions about the desired behavior of the assistant, it works like a general directive. A conversation may therefore look like:

```
[
 {'role': 'system',
  'content': 'You are a concise PyTorch instructor.'},
 {'role': 'user',
  'content': 'Explain gradient descent in one short paragraph.'},
 {'role': 'assistant',
  'content': 'Gradient descent is a fundamental optimization technique used in machine learning to find the minimum of a function. It starts with a starting point, typically a random initial guess, and iteratively updates the solution to the problem by minimizing the difference between the current solution and the target function. This process is repeated until the gradient of the function is zero, indicating that the solution is optimal.'}
]
```

This representation is convenient for humans, but the model doesn't consume Python dictionaries, it all boils down to token IDs. This means we need to transform these structured messages into a sequence of tokens. This is the role of chat templates.

Instruction-tuned models are normally trained using a particular text format. One model might expect something resembling:

```
<|user|>
Hello
<|assistant|>
Hi!
```

Another might expect a different template, such as:

```
[INST] Hello [/INST] Hi!
```

These differences matter. Two instruction models may share the same base architecture yet expect different conversational formats. Providing the wrong format can degrade model behavior substantially. Hugging Face therefore stores model-specific formatting rules as **chat templates** on the tokenizer.

We can inspect the template associated with our tokenizer:

In [13]:
print(tokenizer.chat_template)

{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


In [15]:
messages = [
    {"role": "system",
     "content": "You are a concise PyTorch instructor."},
    {"role": "user",
     "content": "What is backpropagation?"}
]

formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(formatted)

<|im_start|>system
You are a concise PyTorch instructor.<|im_end|>
<|im_start|>user
What is backpropagation?<|im_end|>



In [16]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
inputs

{'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253, 19484, 43943, 13334,    30,
             2,   198,     1,  4093,   198,  1780,   314,  1056, 34901,   309,
            47,     2,   198,     1,   520,  9531,   198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]])}

### 17.4.1 Generation Prompt

In [17]:
without_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
with_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(without_prompt)
print("----------------")
print(with_prompt)

<|im_start|>system
You are a concise PyTorch instructor.<|im_end|>
<|im_start|>user
What is backpropagation?<|im_end|>

----------------
<|im_start|>system
You are a concise PyTorch instructor.<|im_end|>
<|im_start|>user
What is backpropagation?<|im_end|>
<|im_start|>assistant



### 17.4.2 Prefilling

In [19]:
messages = [
    {"role": "user",
     "content": "Return the answer as JSON."},
    {"role": "assistant",
     "content": '{"answer": "'}
]

In [20]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
inputs

{'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
          3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
           198,     1,  4093,   198, 14362,   260,  2988,   347, 20110,    30,
             2,   198,     1,   520,  9531,   198, 39428, 11247,  1799,   476]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

## 17.5 Loading an Instruction Model

In [21]:
checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)
model.eval()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (r

In [22]:
model_f16 = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float16).to(device)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [33]:
def generate_response(messages, max_new_tokens=128, **generation_kwargs):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    prompt_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, **generation_kwargs)

    response_ids = outputs[0, prompt_length:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)

In [38]:
messages = [
    {"role": "user",
     "content": "What does model.eval() do in PyTorch?"}
]
response = generate_response(messages, max_new_tokens=128)
print(response)

In PyTorch, `model.eval()` is a method that evaluates the model's predictions on the test data. It takes two arguments: the model's input tensor and the test data tensor.

The `eval()` method evaluates the model's predictions on the test data, returning the predicted values.

Here's an example:

```python
import torch

# Define a model
model = torch.nn.Linear(10, 5)

# Define the test data
test_data = torch.randn(10, 5)

# Evaluate the model
model.eval()


In [39]:
messages.append(
    {"role": "assistant",
     "content": response}
)
messages.append(
    {"role": "user",
     "content": "Does it disable gradient computation too?"}
)

In [40]:
response = generate_response(messages)
print(response)

No, `model.eval()` does not disable gradient computation. It only evaluates the model's predictions on the test data, and does not perform gradient computation.

However, if you want to perform gradient computation, you can use the `grad` keyword argument in `model.eval()` to specify the type of gradient computation to perform.

Here's an example:

```python
import torch

# Define a model
model = torch.nn.Linear(10, 5)

# Define the test data
test_data = torch.randn(10, 5)




## 17.6 System Messages

In [50]:
messages = [
    {"role": "system",
     "content": "You are a concise PyTorch instructor. Answer shortly, in a single sentence if possible."},
    {"role": "user", 
     "content": "What is gradient accumulation?"}
]
response = generate_response(messages)
print(response)

Gradient accumulation is a technique used in deep learning to compute the gradient of a function, which is a measure of how the function changes as the input changes. It is a fundamental concept in gradient descent, a popular optimization algorithm used in machine learning.


In [52]:
messages[0] = {
    "role": "system",
    "content": "You are teaching an advanced deep learning engineer. Be technically precise.",
}
response = generate_response(messages)
print(response)

Gradient accumulation is a technique used in deep learning to compute the gradient of a loss function with respect to a model's parameters. It is a fundamental concept in deep learning that allows you to compute the gradient of the loss function with respect to the model's parameters, which is essential for training models.

In essence, gradient accumulation is a way to compute the gradient of the loss function with respect to a model's parameters, which is a crucial step in training a model. The gradient of the loss function is a vector of partial derivatives, where each partial derivative represents the gradient of the loss function with respect to a specific parameter.



## 17.7 Generation Controls in Practice

In [53]:
response = generate_response(messages, max_new_tokens=50)
print(response)

Gradient accumulation is a technique used in deep learning to compute the gradient of a loss function with respect to a model's parameters. It is a fundamental concept in deep learning that allows you to compute the gradient of the loss function with respect to the model


In [56]:
response = generate_response(messages, do_sample=False)
print(response)

Gradient accumulation is a technique used in deep learning to compute the gradient of a loss function with respect to a model's parameters. It is a fundamental concept in deep learning that allows you to compute the gradient of the loss function with respect to the model's parameters, which is essential for training models.

In essence, gradient accumulation is a way to compute the gradient of the loss function with respect to a model's parameters, which is a crucial step in training a model. The gradient of the loss function is a vector of partial derivatives, where each partial derivative represents the gradient of the loss function with respect to a specific parameter.



In [57]:
response = generate_response(messages, do_sample=True, temperature=0.8, top_p=0.9)
print(response)

Gradient accumulation is a fundamental concept in neural networks, particularly in the context of optimization and reinforcement learning. It is the process of accumulating the gradient of a loss function in a way that minimizes the loss function or returns the most important information.

In essence, gradient accumulation is a way to compute the gradient of a loss function, which is a way of computing the output of a neural network at each iteration. The gradient represents the rate of change of the loss function with respect to the input values.

Here's a step-by-step example:

1. The loss function is defined as a linear function of the input


In [58]:
response = generate_response(messages, do_sample=True, temperature=0.8, top_p=0.9, repetition_penalty=1.1)
print(response)

Gradient accumulation, also known as gradient descent, is a mathematical optimization technique used in machine learning and data analysis to find the minimum of a function. It's called "gradient" because it accumulates changes in one variable over time or space, like gradients accumulating on a hill. In essence, gradient ascent aims to minimize the difference between a given function and its derivative at each point in space or time, ensuring that the solution converges to a local extremum (global maximum) rather than drifting away from it due to divergence or oscillations.

In simpler terms, imagine you're trying to find the most efficient way to move around your city


### 17.7.1 Generation Configuration 

In [63]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
      max_new_tokens=128,
      do_sample=True,
      temperature=0.8,
      top_p=0.9,
      repetition_penalty=1.05,
      pad_token_id=tokenizer.pad_token_id,
      eos_token_id=tokenizer.eos_token_id
)
with torch.inference_mode():
    outputs = model.generate(**inputs, generation_config=generation_config)

prompt_length = inputs["input_ids"].shape[-1]
response_ids = outputs[0, prompt_length:]
print(tokenizer.decode(response_ids, skip_special_tokens=True))

Gradient accumulation is a fundamental concept in gradient optimization, which is a crucial technique in machine learning and artificial intelligence. It involves computing the gradients of a function with respect to the parameters of a model, which helps to update the parameters based on the learning process.

Given a function f(x) and a vector x_min (or y_min), where y_min is the minimum value of f at the smallest input or sample, gradient accumulation calculates the gradient of f at this point. The resulting gradient is then used to compute the loss function, which measures the difference between the predicted output and the actual output for each iteration


## 17.8 Context Windows and Conversation Length

In [64]:
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
)
n_tokens = len(encoded["input_ids"])
n_tokens

32

## 17.9 Memory and Numerical Precision

In [65]:
n_parameters = model.num_parameters()
n_parameters

134515008

In [66]:
fp32_mb = n_parameters * 4 / 1024**2
fp16_mb = n_parameters * 2 / 1024**2
fp32_mb, fp16_mb

(513.134033203125, 256.5670166015625)

### 17.9.1 Reducing Model's Memory Footprint

In [68]:
model = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float16).to(device)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

## 17.10 Batched Inference

In [69]:
batch_messages = [
    [{"role": "user",
      "content": "What is a tensor?"}],
    [{"role": "user",
      "content": "Why do we call backward() before optimizer.step()?"}],
]

In [70]:
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.unk_token

In [73]:
batch_inputs = (
    tokenizer.apply_chat_template(
        batch_messages,
        tokenize=True,
        add_generation_prompt=True,
        padding=True,
        return_dict=True,
        return_tensors="pt",
    )
    .to(device)
)
batch_inputs["input_ids"], batch_inputs["input_ids"].shape

(tensor([[    2,     2,     2,     2,     2,     2,     2,     1,  9690,   198,
           2683,   359,   253,  5356,  5646, 11173,  3365,  3511,   308, 34519,
             28,  7018,   411,   407, 19712,  8182,     2,   198,     1,  4093,
            198,  1780,   314,   253, 13104,    47,     2,   198,     1,   520,
           9531,   198],
         [    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
           3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
            198,     1,  4093,   198,  4898,   536,   392,   946, 17301,  1000,
           1092, 20882,    30,  5528,  1000,    47,     2,   198,     1,   520,
           9531,   198]]),
 torch.Size([2, 42]))

In [74]:
with torch.inference_mode():
    batch_outputs = model.generate(**batch_inputs, max_new_tokens=80, do_sample=False)

In [75]:
prompt_length = batch_inputs["input_ids"].shape[-1]
generated_ids = batch_outputs[:, prompt_length:]
responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
responses

['A tensor is a fundamental concept in mathematics, particularly in the realm of linear algebra and mathematical analysis. It is a mathematical object that represents a multi-dimensional array of numbers, known as a vector. In essence, a tensor is a way to represent a vector as a matrix, where each element of the vector is a scalar value.\n\nThink of a tensor as a matrix with a specific structure',
 "The `backward()` method in Python is used to reverse the order of the elements in a list. It is called `backward` because it reverses the order of the elements in the list.\n\nHere's a simple example:\n\n```python\nnumbers = [1, 2, 3, 4, 5]\n\n# Using backward()\nnumbers.backward()"]

## 17.11 High-Level Inference with Pipelines

In [76]:
from transformers import pipeline

generator = pipeline(task="text-generation", model=checkpoint, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [79]:
messages = [
    {"role": "system",
     "content": "You are a concise PyTorch instructor."},
    {"role": "user",
     "content": "What is gradient clipping?"},
]
output = generator(messages)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [82]:
print(output[0]["generated_text"][-1]["content"])

Gradient clipping is a technique used in PyTorch to selectively preserve the order of gradients in a network. It is essentially a way to remove gradients that are not contributing to the model's cost function.

In practice, gradient clipping involves dividing the gradients in the direction of the direction of the gradient of the loss function. This is done by subtracting the corresponding gradient from the gradient of the loss function in the direction of the direction of the gradient, but in reverse order in the direction of the gradient.

For example, if we have a network with a loss function with a gradient of `w1` in the direction of `w2`, we can use gradient clipping by subtracting the gradient of `w1` in the direction of `w2`:

```python
# Original network
x = torch.rand(10, 20, 3)
W = torch.rand(2, 2) + (torch.randn(-1, 2)) / 2

# Gradient clipping
x_clip = x - W * x
W_clip = W - x_clip * x

# Output
x_clip = x.view(-1, 10, 2)
W_clip


In [85]:
messages = output[0]["generated_text"]
messages.append({
    "role": "user",
    "content": "When would I use it?",
})
output = generator(messages)
print(output[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Gradient clipping is used in various applications, including:

1. **Network optimization**: Gradient clipping is used in network optimization to select the most promising network configurations, as it can help to maintain the order of gradients in the optimization process.
2. **Model selection**: Gradient clipping can be used to select the best model parameters during model selection, as it can help to avoid unnecessary gradients.
3. **Model-agnostic optimization**: Gradient clipping is a general technique that can be applied to any optimization problem, regardless of the specific model or optimizer being used.

The choice of when to use gradient clipping depends on the specific problem and the goals of the model or algorithm being optimized. Some examples of use cases include:

* Network optimization: to select the best network configuration during optimization.
* Model selection: to select the best model parameters during model selection.
* Model-agnostic optimization: to select the 

## 17.12 Choosing an LLM Checkpoint

We've used one small model throughout these chapters, but the Hugging Face Hub contains many language-model families and many variants of each family. Choosing the right repository requires more than picking the largest model that fits in memory.

### 17.12.1 Base or Instruct?

First decide what behavior you need. For ordinary conversational or instruction-following, instruction or chat models are generally the appropriate starting point. A base model is more useful when:
- studying pretraining behavior;
- performing specialized continued pretraining;
- building a custom instruction-tuning pipeline;
- researching language-model behavior before alignment.

The model repository name often makes the distinction explicit, but always verify it in the model card.

### 17.12.2 Parameter Count

Parameter count affects both capability and hardware requirements. Broadly speaking, the more parameters a model has, the more memory its weights will take, and more computation will generally be performed. But parameter count alone does not determine the model's quality: training data, architecture, training objective, and post-training matter enormously. A newer well-trained small model may outperform a much larger older one on particular tasks.

### 17.12.3 Context Length

Don't forget to check how much context the model supports. This matters for applications involving:
- long conversations;
- large documents;
- codebases;
- many examples in the prompt.

Keep in mind that a longer advertised context also means potentially larger KV caches and greater inference cost.

### 17.12.4 Architecture

Model cards and configurations may expose architectural choices such as:
- number of layers;
- hidden dimensions;
- attention heads;
- key/value heads;
- RoPE configuration;
- sliding-window attention;
- vocabulary size.

You don't need to select models based on every low-level detail, but understanding these fields helps explain differences in memory and behavior.

### 17.12.5 Chat Template

For an instruction model, check that its tokenizer provides an appropriate chat template. If no template is included, you'll need reliable documentation describing the expected prompt format. Do not simply invent role markers, because the model performs best when inference reproduces the structure it encountered during instruction tuning.

### 17.12.6 License

Model weights also have licenses. Two technically similar models may permit very different uses. Always inspect the model card and license when choosing a model for a real application. The SmolLM2 model we're using is distributed under Apache 2.0.

### 17.12.7 Model Card

The model card should be one of your first stops. Look for information about:
- intended use;
- architecture;
- training;
- languages;
- context length;
- inference examples;
- evaluation;
- known limitations;
- license.

The SmolLM2 card, for example, states that the family is primarily focused on English and reports separate evaluations for the base and instruction variants.

### 17.12.8 Hardware Requirements

Finally, estimate whether the model actually fits your hardware. At a minimum, consider the space the weights themselves take, but leave room for KV cache, temporary tensors, framework overhead, etc. When the model doesn't fit comfortably, techniques such as reduced precision, quantization, and parameter-efficient fine-tuning become important. Which brings us directly to the next chapter.

## 17.13 What Instruction Tuning Actually Changed

We've covered a lot of practical machinery. Let's return to the core idea: our base model from Chapter 16 was trained primarily through causal language modeling, that is, predicting the next token. Instruction tuning did **not** replace that decoder architecture with a fundamentally different model, it simply exposed the pretrained model to specially structured examples demonstrating desirable interactions. Through training on such examples, the model learns that after tokens representing a user request and an assistant marker, desirable next tokens resemble a useful response. At inference time, we reproduce the conversational structure using the model's chat template, which preserves the expected structure.

## 17.18 Looking Ahead

We started Part IV by representing text. In Chapter 15, we covered embeddings, similarity, and retrieval. In Chapter 16, we moved to generation, discussing the causal Transformer, next-token prediction, and autoregressive generation. In this chapter, we added instruction-following behavior to the mix and and learned how to use the structure provided by a chat template. We also saw that a conversation isn't secretly stored inside the model. It is the application that maintains the "memory" in the message list and sends the relevant history back to the model on every turn. We revisited generation controls in a practical setting, examined how context grows during conversation, estimated memory requirements, batched several requests, and finally replaced our explicit inference code with Hugging Face's high-level chat pipeline.

There is one major question left: how did the base model become an instruction model in the first place? Training every parameter in a modern LLM can require enormous amounts of GPU memory. Fortunately, we often don't need to. In the next chapter, **Efficient Fine-Tuning of LLMs with LoRA and QLoRA**, we'll learn how to adapt pretrained models efficiently, changing their behavior without retraining billions of parameters.